# Brain Map Classification Pipeline

## Pipeline Overview

1. **LLM Categorization**: Use Claude to categorize ~2,458 ngram labels into 5 categories
2. **Dataset Creation**: Build paper-level labeled dataset

## Target Categories
- Networks: DMN, salience network, etc.
- Regions: precuneus, hippocampus, etc.
- Cognitive_Functions: working memory, attention, etc.
- Tasks: n-back, Stroop, etc.
- Conditions: depression, schizophrenia, etc.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Part 1: Data Inspection

Let's first load and inspect the raw ngram data.

In [ ]:
# Load ngram data
labels = np.load('ngram_labels.npy', allow_pickle=True)
matrix = np.load('ngram_matrix.npy', allow_pickle=True)

with open('pmids.txt', 'r') as f:
    pmids = [line.strip() for line in f if line.strip() != 'pmid']

print(f"Number of labels: {len(labels)}")
print(f"Matrix shape: {matrix.shape}")
print(f"Number of PMIDs: {len(pmids)}")
print(f"\nMatrix sparsity: {(matrix == 0).sum() / matrix.size:.2%}")
print(f"Average labels per paper: {matrix.sum(axis=1).mean():.2f}")

In [ ]:
# Sample labels
print("Sample labels:")
sample_indices = np.random.choice(len(labels), 20, replace=False)
for idx in sorted(sample_indices):
    print(f"  {labels[idx]}")

In [ ]:
# Distribution of labels per paper
labels_per_paper = matrix.sum(axis=1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(labels_per_paper, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Number of labels per paper')
plt.ylabel('Frequency')
plt.title('Distribution of Labels per Paper')

plt.subplot(1, 2, 2)
papers_per_label = matrix.sum(axis=0)
plt.hist(papers_per_label, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Number of papers per label')
plt.ylabel('Frequency')
plt.title('Distribution of Papers per Label')

plt.tight_layout()
plt.show()

## Part 2: LLM Label Categorization

In [ ]:
# Load categorization results
import json

with open('label_categories.json', 'r') as f:
    label_categories = json.load(f)

print(f"Categorized labels: {len(label_categories)}")

# Analyze category distribution
categories = ["Networks", "Regions", "Cognitive_Functions", "Tasks", "Conditions"]
category_counts = {cat: 0 for cat in categories}

for label, cats in label_categories.items():
    for cat, is_member in cats.items():
        if is_member:
            category_counts[cat] += 1

print("\nLabels per category:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count} ({count/len(label_categories)*100:.1f}%)")

In [ ]:
# Visualize category distribution
plt.figure(figsize=(10, 6))
plt.bar(category_counts.keys(), category_counts.values(), alpha=0.7, edgecolor='black')
plt.xlabel('Category')
plt.ylabel('Number of Labels')
plt.title('Distribution of Labels Across Categories')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Show example labels from each category
print("Example labels per category:\n")
for cat in categories:
    examples = [label for label, cats in label_categories.items() if cats.get(cat, False)][:10]
    print(f"{cat}:")
    for ex in examples:
        print(f"  - {ex}")
    print()